# Sorts and Long-Short Portfolios
## 🎯 Learning Objectives

By the end of today you will be able to:

1. **Turn any firm characteristic into a portfolio** using the sort recipe —
   rank, bucket, weight, average, spread
2. **Build a long-short portfolio** and explain why its weights sum to zero
3. **Compare the equal- and value-weighted versions** of the same sort, and say
   why they differ
4. **Check what a sort did to the signal itself** — how far apart the
   portfolios really are
5. **Recognize when a spread is an artifact of your implementation** rather than
   a property of the signal

## 📋 Today's Plan

1. [The sort recipe](#recipe)
2. [Pitfall checklist](#pitfalls)
3. [🔄 Live Demo: sorting on value](#demo)
4. [Why ten buckets?](#tradeoff)
5. [🛠️ Hands-On: pick your own signal](#ho1)
6. [🎯 Challenge: your signal, two ways](#challenge) — *homework*
7. [Key takeaways](#takeaways)

> **📎 Winsorizing and z-scoring** live in the Appendix —
> `chapters/Appendix/SignalHygiene_AI.ipynb`. Ranking is scale-free so a sort
> doesn't need them, but you will if you combine two signals for Assignment 1.

---

## 🛠️ Setup

In [ ]:
#@title Setup — run this first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [11, 4.5]
plt.rcParams['font.size'] = 11
import warnings; warnings.filterwarnings('ignore')

BASE = "https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data"
panel = pd.read_parquet(f"{BASE}/panel_backbone_1980_2000.parquet")
menu  = pd.read_csv(f"{BASE}/signal_menu.csv")

print(f"panel: {len(panel):,} rows, {panel.date.nunique()} months")
print(f"menu : {len(menu)} signals available\n")
menu[['Acronym', 'Authors', 'Year', 'Cat.Economic']].head(30)

---

## 1. The Sort Recipe <a id="recipe"></a>

Last week you built one portfolio holding everything. Today you build a
portfolio that expresses a **view**.

The setup: you have a number for every stock in every month — a *signal*. It can
come from accounting data, from past returns, from the text of earnings calls,
from ownership records, from short interest, from satellite images of parking
lots. **Anything.** Three things have to be true and nothing else does: you have
a value for every stock on every date, it varies across firms, and you have some
theory of how it relates to expected returns.

You believe stocks with a high value of that number will outperform stocks with
a low value. How do you turn that belief into a portfolio?

> **The recipe — five steps, and it never changes**
>
> 1. **Rank** every stock each month by the signal — *within* that month, never
>    across the whole sample. You are grading on a curve, by cohort
> 2. **Bucket** them into groups (deciles is standard)
> 3. **Weight** within each bucket (equal or value)
> 4. **Average** each bucket's `ret` to get that bucket's return — the signal
>    and the weights are from *last* month, the return is *this* month
> 5. **Spread** — go long the top bucket, short the bottom

Every factor you have ever heard of — value, momentum, quality, low-volatility
— is this recipe applied to a different column. That is genuinely all there is
to it.

### Why decile portfolios and not just a regression?

You could regress returns on the signal. Sorting has three advantages that
matter in practice:

- It is **non-parametric** — no assumption that the relationship is linear
- It shows you the **shape** — is the effect in the extremes, or monotone?
- The output **is a portfolio**, with a return you could actually have earned

### Step 5 is a self-financed position

Long the top decile, short the bottom, in equal dollars. The weights sum to
**zero** — exactly the structure you met in Lecture 1 when you computed an
excess return. It costs nothing to enter, so what comes out is a *spread*, not
a return on capital.

Insisting that each leg's weights sum to one is what gives you a clear notion of
**book size**. If you run the long-short at 100×, you know exactly what that
means: buy \$100 of the long leg, sell \$100 of the short. Without that
normalization the number you report is scaled by an arbitrary constant and is
not comparable to anyone else's.

### From names to characteristics

This is the move that defines quantitative investing.

A discretionary analyst says *"I like Apple, because tariffs will hurt its
competitors more."* A quant says *"build a tariff-exposure characteristic, sort
every stock on it, and hold the whole top decile."*

The second version is the same idea, made diversified and repeatable. You are
no longer betting on a company; you are betting on a **property**, held across
hundreds of companies at once, and you find out whether the property pays.

> **💡 Key Insight: the portfolios churn, and that's the point**
>
> Sorting doesn't buy a fixed list of firms. Membership turns over as companies
> change, which is exactly what makes it a bet on the characteristic rather than
> on the names.
>
> Microsoft is the standard illustration. Small in the early 80s, so it sat in
> the small-cap bucket. Gigantic by the late 90s, so it moved to the large-cap
> bucket. Priced enormously above book value during the tech boom → the *growth*
> (low book-to-market) portfolio. Valuation collapsed after 2000 → it migrated
> into the *value* portfolio. Now, with AI, it's back in growth.
>
> One company. Four decades. It has been in nearly every bucket we sort on. A
> value strategy held it when it was cheap and dropped it when it wasn't —
> without anyone forming an opinion about Microsoft.

But the recipe is not magic, and it is worth knowing what failure looks like.
Sort stocks into 26 portfolios by the first letter of the ticker and you will
get no spread in average returns at all — each portfolio will look like the
market, only more volatile, because you have thrown away diversification to buy
nothing. The machinery runs identically either way. **The characteristic has to
capture some real economics**; the sort only harvests what is already there.

---

## 🛡️ Pitfall Checklist for Sorts <a id="pitfalls"></a>

| | Pitfall | What goes wrong | 🔍 How to detect |
|---|---|---|---|
| 1 | **Sorting on the signal at *t*, measuring `ret` at *t*** | Look-ahead. Inflates everything, sometimes enormously | Is the signal lagged relative to the return? |
| 2 | **Equal-count deciles over all stocks** | The bottom decile becomes ~600 microcaps that no one could trade | How many stocks are in the extreme buckets, and how big are they? |
| 3 | **Equal-weighting inside buckets** | Tiny firms get the same weight as Apple | Is a `weights=` argument present? |
| 4 | **Ranking across the whole panel instead of within each month** | You compare 1985's book-to-market to 1999's | Is `groupby('date')` in the ranking step? |
| 5 | **Dropping missing signal values after bucketing** | Bucket sizes become unequal  | Drop first, then rank |
| 6 | **Reporting only the spread** | A "great" long-short may be entirely a bad short leg you couldn't execute | Always print both legs separately |

> **🤖 AI-Era Insight**
>
> Ask for "a decile sort on this signal" and you will get `pd.qcut(x, 10)` over
> the full cross-section, equal-weighted, every time. That is pitfalls 2 and 3
> together. Today's challenge measures what pitfall 3 does to your own signal.
> Assignment 2 takes on pitfall 2: the fix research papers use, NYSE
> breakpoints, and what it does to the size effect — it flips the sign.

---

## 🔄 Live Demo: Sorting on Value <a id="demo"></a>

Book-to-market — book value divided by market value. High means the market
prices the firm cheaply relative to its accounting value. Graham and Dodd in
1934, Stattman in 1980, Fama and French in 1992.

### Step 1 — The data

The panel from last week has returns and market caps but no accounting data.
The signals live in separate files, one per signal, in the course repo:

```
{BASE}/signals/<Acronym>.parquet
```

There are 30 of them, and the `menu` table the Setup cell printed lists them
all. Every file has the same three columns: `permno`, `date`, and the signal,
named after its acronym. Book-to-market is `BM`.

In [ ]:
#@title Load the book-to-market signal
bm = pd.read_parquet(f"{BASE}/signals/BM.parquet")

print(f"{len(bm):,} stock-months, {bm.permno.nunique():,} firms, "
      f"{bm.date.min():%b %Y} to {bm.date.max():%b %Y}")
print(menu.loc[menu.Acronym == 'BM', 'LongDescription'].iloc[0])
bm.head()

In [ ]:
#@title What is in it
# 1. It is a log. np.exp() turns it back into book value / market value.
print("BM across all stock-months")
for q in [0.1, 0.5, 0.9]:
    v = bm['BM'].quantile(q)
    print(f"  {q:.0%} percentile: BM = {v:+.2f}   book / market = {np.exp(v):.2f}")

# 2. It changes once a year. General Electric's fiscal year ends in December.
ge = bm[bm.permno == 12060].set_index('date')['BM']
print("\nGE: the months in which its BM changes")
print(ge[ge.diff() != 0].iloc[1:7].round(3).to_string())   # row 0 is just where the sample starts

# 3. Not every stock has one.
has_bm = panel.merge(bm, on=['permno', 'date'], how='left')['BM'].notna()
print(f"\nShare of panel stock-months with a BM: {has_bm.mean():.0%}")

### What that tells you

- **It is a log.** `BM` is log(book value ÷ market value), so zero means the two
  are equal and a negative number means the market values the firm above its
  books. The median stock-month is about −0.58: book value a bit over half of
  market value. High `BM` is *value*, low `BM` is *growth*.
- **It changes once a year.** Book value comes from the annual report, and both
  halves of the ratio are fixed at that report — `BM` does not move when the
  price moves during the year. GE's fiscal year ends in December and its `BM`
  updates every June, because Open Source Asset Pricing, where these files come
  from, only uses a report six months after the year it covers has ended.
- **About a quarter of stock-months have none.** Roughly half of those are young
  firms whose first annual report isn't public yet. They are simply left out of
  the sort.

A row dated *t* holds what was known at the end of month *t*. To earn this
month's `ret`, sort on **last month's** `BM` — lag it one month within each
stock, exactly as you lagged `me` last week.

### Step 2 — The AI prompt (try step by step)

>
> 1. Merge the `BM` signal onto the panel and lag it one month within each stock.
> 2. Drop rows with missing market cap, signal, andreturns.
> 2. Rank stocks by the lagged `BM` and cut into 10 equal-count buckets. 
> 3. For each bucket and month, compute the equal-weighted average of returns. 
> 4. Show a plot of the cumuatlive returns of each portfolio
> 5. Show a plot of the cumulative retruns of the long-short that is long decile 10 and short decile 1
> 6. Report each decile's annualized mean return and the annualized decile-10-minus-decile-1 spread with its t-statistic. 

In [ ]:
# write your work here

In [ ]:
#@title 🔒 Reference — run after yours; the cells below use its `bucket` and `ls`
panel['me_l1'] = panel.groupby('permno')['me'].shift(1)

d = panel.merge(bm, on=['permno', 'date'], how='left').sort_values(['permno', 'date'])
d['BM_l1'] = d.groupby('permno')['BM'].shift(1)          # last month's BM
d = d.dropna(subset=['BM_l1', 'ret', 'me_l1'])

# rank WITHIN each month (pitfall 4)
d['decile'] = d.groupby('date')['BM_l1'].transform(
    lambda x: pd.qcut(x, 10, labels=False, duplicates='drop'))

bucket = d.groupby(['date', 'decile'])['ret'].mean().unstack()
ls = (bucket[9] - bucket[0]).dropna()

print("Annualized mean return by book-to-market decile (1 = growth, 10 = value)")
for k in range(10):
    print(f"  D{k+1:<2d} {bucket[k].mean()*12:7.2%}")
print(f"\nD10 - D1 : {ls.mean()*12:.2%}/yr")
print(f"t-stat   : {ls.mean()/ls.std()*np.sqrt(len(ls)):.2f}")
print(f"months   : {len(ls)}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(range(1,11), bucket.mean()*12, color='steelblue')
ax[0].set_xlabel('Book-to-market decile'); ax[0].set_ylabel('Mean return, annualized')
ax[0].set_title('The sort', fontweight='bold')
ax[1].plot(ls.index, (1+ls).cumprod(), color='darkorange', linewidth=1.8)
ax[1].set_ylabel('Growth of $1'); ax[1].set_title('Long-short D10 - D1', fontweight='bold')
plt.tight_layout(); plt.show()

### Step 3 — Validate

> **🤔 Read the bar chart before the t-stat.**
>
> Is the pattern **monotone** — does each decile beat the one below it? 
> What to make of it?

Also check both legs separately (pitfall 6): a spread built entirely from a
disastrous short leg is a different claim from one where both sides contribute. Why that matters?

---

## 2. Why Ten Buckets? <a id="tradeoff"></a>

Why deciles? Why not 100 buckets, so the top portfolio has a far more extreme
signal? Or simply buy the single highest-signal stock?

Because a portfolio return is an *estimate*, and estimates have standard errors.

Individual stocks have annual volatility of **40–80%**. To measure an expected
return to within a percentage point you need $\sigma/\sqrt{T}$ small — and with
σ = 60%, that takes centuries. Averaging across stocks in a bucket cuts the
volatility fast, which is what makes the bucket's mean return measurable at all.

So there is a genuine trade-off:

| More buckets | Fewer buckets |
|---|---|
| Stronger signal in the extremes | Weaker signal |
| Fewer stocks each → noisier | More stocks → more precise |
| Higher turnover, worse capacity | Cheaper to trade |

Ten is a convention that balances these. How would you check if it is a good one?

In [ ]:
for n in [5, 10, 20, 50]:
    x = panel.merge(bm, on=['permno','date'], how='left').sort_values(['permno','date'])
    x['BM_l1'] = x.groupby('permno')['BM'].shift(1)
    x = x.dropna(subset=['BM_l1', 'ret', 'me_l1'])
    x['g'] = x.groupby('date')['BM_l1'].transform(
        lambda v: pd.qcut(v, n, labels=False, duplicates='drop'))
    p = x.groupby(['date','g'])['ret'].mean().unstack()
    r = (p[n-1] - p[0]).dropna()
    n_stocks = x[x.g == n-1].groupby('date').size().mean()
    print(f"{n:>3d} buckets: spread {r.mean()*12:>6.1%}/yr   "
          f"t = {r.mean()/r.std()*np.sqrt(len(r)):>4.1f}   "
          f"{n_stocks:>5.0f} stocks in the top bucket")

> **💡 Key Insight**
>
> Cutting finer **does** widen the spread — monotonically, from 16% at quintiles
> to 29% at 50 buckets. The top 2% of stocks by book-to-market really are
> cheaper than the top 10%, and they really do earn more.
>
> But look at the t-statistic. It rises to about 7.3 at 20 buckets and then
> **falls**. Past that point each bucket holds so few stocks that the extra
> signal is swamped by the extra noise. This noise is not only impacting t-stat--but it is real risk that you have to bear. 


---

## 🛠️ Hands-On: Pick Your Own Signal <a id="ho1"></a>

You have **30 signals**, each from a published paper, each with the t-statistic
the original authors reported. Pick one and test it.

> **📌 Everyone uses the demo's construction** so the class results are
> comparable: the signal lagged one month, ten equal-count buckets within each
> month, top decile minus bottom decile. Assignment 2 moves you to the
> convention research papers use.

> **⚠️ Three signals won't cut into ten equal-count buckets.** `DivSeason` and
> `OScore` take only two values, and `ShareIss1Y` is zero for so many firms that
> most months end up with nine buckets. Pick another one today.

### Your task

1. Pick a signal from the menu
2. Run the sort
3. Report your t-statistic next to the published one
4. Be ready to say out loud whether it replicated

How would you make a combo?

In [ ]:
# The menu — 30 published signals
pd.set_option('display.max_rows', 40, 'display.width', 200)
menu[['Acronym','Authors','Year','Cat.Economic','T-Stat']].sort_values('Cat.Economic')

In [ ]:
# === EDIT THIS CELL: your signal ===
MY_SIGNAL = "GP"      # ← pick any Acronym from the menu above

sig = pd.read_parquet(f"{BASE}/signals/{MY_SIGNAL}.parquet")
print(f"{MY_SIGNAL}: {len(sig):,} stock-months")
print(menu.loc[menu.Acronym == MY_SIGNAL, 'LongDescription'].iloc[0][:300])

In [ ]:
# your turn:

### What did the room find?

Compare with your neighbours. Some of you have a t-statistic above 4. Some have
one near zero on a signal whose authors reported 8.

Both are honest results from the same code.

> **🤔 Before we move on**
>
> Your sample is 1980–2000. Look up the sample period the original paper used. If your window barely overlaps theirs, what exactly have you
> tested? And if a published t-statistic of 8 becomes 1 in a different window,
> which number should you believe?

Hold that question. It is the backtesting and anomalies lectures (L8 and L9), and for several of you it
will be your project.

---

## 🎯 Challenge: Your Signal, Two Ways <a id="challenge"></a>

Take the signal you picked in the Hands-On — `MY_SIGNAL` — and sort on it the way
the demo sorted on value: the signal lagged one month, rows missing `ret`,
`me_l1` or the signal dropped first, ten equal-count buckets within each month,
top decile minus bottom.

Then weight it two ways, and look at what the sort did to the signal itself.

### Q1 — Equal-weighted and value-weighted

Build the D10 − D1 long-short twice: once with every stock in a decile weighted
equally, once weighted by last month's market cap, `me_l1`. For each, report the
annualized mean return and the t-statistic.

> **📌 Required variable names:**
> ```python
> ls_ew_ann = ____   # equal-weighted D10 - D1, annualized mean return
> t_ew      = ____   # its t-statistic
> ls_vw_ann = ____   # value-weighted D10 - D1, annualized mean return
> t_vw      = ____   # its t-statistic
> ```

In [ ]:
# Your work here


# Required outputs — fill these in:
ls_ew_ann = ____
t_ew      = ____
ls_vw_ann = ____
t_vw      = ____

print(f"{MY_SIGNAL}, D10 - D1, 1980-2000")
print(f"  equal-weighted : {ls_ew_ann:+.2%}/yr   t = {t_ew:+.2f}")
print(f"  value-weighted : {ls_vw_ann:+.2%}/yr   t = {t_vw:+.2f}")

### Q2 — How far apart are the portfolios on the signal?

A sort promises that decile 10 holds high values of the signal and decile 1 low
ones. Check how far apart they really are. Each month, average the lagged signal
within each decile; then average those ten numbers over the months.

Plot them as a bar chart, next to a bar chart of the ten deciles' mean returns.

> **📌 Required variable names:**
> ```python
> sig_d1  = ____   # average lagged signal in decile 1, the bottom
> sig_d10 = ____   # average lagged signal in decile 10, the top
> ```

In [ ]:
# Your work here


# Required outputs — fill these in:
sig_d1  = ____
sig_d10 = ____

print(f"{MY_SIGNAL}: average lagged signal  D1 = {sig_d1:.4g}   D10 = {sig_d10:.4g}")

### Q3 — The memo

> **📝 Maximum 6 sentences**
>
> Write to your PM about your signal.
>
> A strong memo will:
> - Say whether the signal works, and under which weighting
> - Explain why the equal- and value-weighted answers differ: which stocks are
>   doing the work in each
> - Say what the signal chart shows. Is the signal spread evenly across the ten
>   deciles, or are the extreme deciles far from the rest?
> - Compare the shape of the return chart with the shape of the signal chart

In [ ]:
MEMO = """
Write your memo here. Don't delete the surrounding triple quotes.
"""
print(MEMO)

---

## 📤 Submission <a id="submit"></a>

In [ ]:
# === 📤 SUBMISSION CELL — Run this last ===
import json, base64, hashlib, datetime as dt

required = ["MY_SIGNAL", "ls_ew_ann", "t_ew", "ls_vw_ann", "t_vw",
            "sig_d1", "sig_d10", "MEMO"]
missing = [v for v in required if v not in globals()]
if missing:
    raise NameError(f"\n❌ Missing before submission: {missing}")

payload = {
    "assignment": "L3_Sorts_AI",
    "ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
    "signal": MY_SIGNAL,
    "answers": {k: float(eval(k)) for k in required if k not in ("MY_SIGNAL", "MEMO")},
    "memo": MEMO.strip(),
}
blob = json.dumps(payload, sort_keys=True)
checksum = hashlib.sha256(blob.encode()).hexdigest()[:8]
token = f"UG54::{checksum}::{base64.b64encode(blob.encode()).decode()}"

print("=" * 72)
print("📋  COPY THE LINE BELOW AND PASTE INTO THE SUBMISSION FORM")
print("=" * 72)
print(token)
print("=" * 72)
print(f"\nLength: {len(token)} chars")
print("Submission form: https://forms.gle/yazZ8bbatL87jdJi7")

---

## 🧠 Key Takeaways <a id="takeaways"></a>

1. **The sort recipe is five steps and never changes:** rank, bucket, weight,
   average, spread. Every factor you've heard of is this applied to a different
   column.

2. **Rank within each month**, never across the whole panel.

3. **A long-short portfolio is self-financed** — weights sum to zero, no capital
   tied up, so no risk-free rate to subtract.

4. **Read the monotonicity, not just the spread.** A clean staircase across
   deciles is much stronger evidence than two unusual extremes.

5. **Equal- and value-weighting can give different answers.** Most stocks are
   small, so an equal-weighted sort is mostly a sort of small stocks. Always ask
   how the portfolio was weighted.

6. **Check what the sort did to the signal.** The extreme deciles hold the tails
   of the signal, and they can sit far from everything else.

   The convention research papers use, NYSE breakpoints, is Assignment 2 — along
   with what it does to the size effect.

7. **A spread is not yet a result.** You have a return series; you do not yet
   know whether it is skill, market exposure, or noise. Measuring that starts
   Wednesday.

---

### Next class

Where the data really comes from — CRSP, Compustat, and how you'd pull it
yourself — and then factor models: the tool that separates "this strategy earns
a return" from "this strategy earns a return *you couldn't have got for free*.

---

## 📎 Appendix <a id="appendix"></a>

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 📎 APPENDIX — Belt-and-Suspenders Data Loading
# ═══════════════════════════════════════════════════════════════════════
# Everything today loads from the repo — no WRDS account needed.
#
#   panel  = pd.read_parquet(f"{BASE}/panel_backbone_1980_2000.parquet")
#   signal = pd.read_parquet(f"{BASE}/signals/<Acronym>.parquet")
#   menu   = pd.read_csv(f"{BASE}/signal_menu.csv")
#
# The 30 signals come from Open Source Asset Pricing (Chen & Zimmermann),
# which replicates 300+ published cross-sectional predictors:
#     https://www.openassetpricing.com
#
# TWO CONVENTIONS BAKED INTO THE SIGNAL FILES — worth knowing:
#
# 1. PRE-SIGNED. OSAP ships the raw characteristic plus a Sign column; 18 of
#    our 30 have Sign = -1. We flipped those on write, so for every signal
#    here HIGH = predicted HIGH return and the long-short is always D10 - D1.
#    The original direction is in signal_menu.csv.
#
# 2. Lag the signal. Sorting on a signal at t and measuring ret at t is
#    look-ahead. For short-term reversal that single error turns a t-stat
#    of -0.4 into +70.
#
# To rebuild any of this from scratch: chapters/Finance/build_course_panel.py
